## GPU Computing - Introduction

### ****Milestone 1:** Mandelbrot Float32 Kernel (35 min)**

****Follow** `mandelbrot_tutorial.md` (attached on Moodle). Type the code yourself — that is the goal. The tutorial walks through every step. If you get stuck, peek at `mandelbrot_opencl.py` (don’t open until your own version runs).**

In [ ]:
# From slide 9

# Kernel signature -- fill in the body following the tutorial
KERNEL_SRC = """
__kernel void mandelbrot(
    __global int *result,
    const float x_min, const float x_max,
    const float y_min, const float y_max,
    const int N, const int max_iter)
{
    int col = get_global_id(0);
    int row = get_global_id(1);
    // ... your escape-time loop here ...
}"""

# Host launch -- note np.float32/np.int32 wrapping for every scalar
prog.mandelbrot(
    queue, (N, N), None, image_dev,
    np.float32(X_MIN), np.float32(X_MAX),
    np.float32(Y_MIN), np.float32(Y_MAX),
    np.int32(N), np.int32(MAX_ITER),
)
queue.finish()

- **The kernel is a direct translation of your Numba escape-time loop into OpenCL C**

- **Each work-item computes its own *c* from its pixel coordinates — no input array needed**

- **Launch with 2D global size `(N, N)`; wrap every scalar as `np.float32(...)/np.int32(...)`**

- **Visualise with `plt.imshow` and save the image**

****After it produces a correct image, time it:****

- **Add a warm-up call first (first launch triggers kernel JIT compile)**

- **Time with `time.perf_counter()`; call `queue.finish()` before stopping the clock**

****Done?** Record GPU f32 runtime in performance notebook (MP3) → commit**

### ****Milestone 2:** Float32 vs Float64 (20 min)**

****Task:** Add a float64 version and compare speed and image quality.**

In [ ]:
# From slide 11

# 1. Check fp64 support
dev = ctx.devices[0]
if 'cl_khr_fp64' not in dev.extensions:
    print("No native fp64 -- Apple Silicon: emulated, expect large slowdown")

# 2. Add this pragma at the top of your f64 kernel string
#pragma OPENCL EXTENSION cl_khr_fp64 : enable

# 3. Three changes inside the kernel:
#    float  -> double
#    0.0f, 4.0f -> 0.0, 4.0     (drop the f suffix)
#     (float)N  -> (double)N

# 4. Wrap scalars as np.float64(...) in the host launch call

1. **Check that your device supports fp64 (see code example)**

2. **Copy your M1 kernel; rename to `mandelbrot_f64`; change: `float` → `double`, literal suffixes `0.0f/4.0f → 0.0/4.0`; add the pragma line at the top**

3. **Time both at $N=1024$ and $N=2048$ — what is the f32/f64 speed ratio?**

- **Is the measured speed ratio consistent with your Roofline prediction?**

- **Apple Silicon (M-series): fp64 is emulated in software — expect a very large slowdown**

****Done?** Record f32 and f64 runtimes in performance notebook (MP3) → commit**

### ****Milestone 3:** Benchmark GPU**

****Task:** Time your GPU kernels at the same grid sizes you used in MP1 and MP2. Reuse the CPU runtimes already in your performance notebook — no need to re-run them.**

In [ ]:
# From slide 13

import statistics, time, matplotlib.pyplot as plt
def timed(fn, runs=3):
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(); queue.finish()     # queue.finish() waits for GPU
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)

N = 1024                        # match your MP1/MP2 benchmark N
results = {
    "GPU f32": timed(lambda: run_mandelbrot_f32(N)),
    "GPU f64": timed(lambda: run_mandelbrot_f64(N)),
    # Paste your MP1/MP2 runtimes here:
    # "Naive": ..., "NumPy": ..., "Numba": ...,
}
names, times = zip(*results.items())
plt.bar(names, times, log=True)
plt.ylabel("seconds (log scale)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("benchmark_mp3.png", dpi=150)

1. **Time `mandelbrot_gpu_f32` and `mandelbrot_gpu_f64` at your benchmark *N* values**

2. **Combine GPU runtimes with your recorded MP1/MP2 numbers**

3. **Plot as a bar chart with a log-scale y -axis — this is the MP3 performance comparison figure**

- **Include: naive Python, NumPy, Numba f32/f64, multiprocessing, Dask local, Dask cluster, GPU f32, GPU f64**

- **Where does GPU sit relative to Numba? To naive Python? (Recall the L01 starting point.)**

****Done?** Combined benchmark chart in performance notebook (MP3) → commit**